# 2g-deep — Poisson Jacobi $\Leftrightarrow [\pi,\pi]_{SN}=0$ (engine, sıfırdan)

Bu notebook **fonksiyon-düzeyinde** Poisson cyclic Jacobi obstruction'ını
hiçbir hazır teorem (Derived Bracket Theorem dâhil) cite etmeden, yalnızca
Faz 13.E'nin iki yeni aksiyomuyla evrensel SN engelinin
``BracketApply([·,·]_SN, π, π)`` handle'ına indirger:

| Aksiyom | Tanım |
|---------|-------|
| **2g-1** `{f,g}_π → X_f(g)` | `PoissonAsHamiltonianDefinition` (Faz 13.E) |
| **2g-2** `Σ_cyc X_a(X_b(c)) → ½⟨[π,π]_SN, df∧dg∧dh⟩` | `HamiltonianCyclicSnFormulaDefinition` (Faz 13.E) |

13.D ile (form-level, 2f-deep) yapısal kardeş: aynı evrensel obstrüksiyona
inen iki paralel zincir. Form-tarafı 1-formlar üzerinden Koszul bracket'iyle
açılırken, fonksiyon-tarafı doğrudan Hamiltonian vektör alanlarının
yinelemeli derivasyonu üzerinden ilerler — daha az ara katman, daha keskin
kapanış.

In [1]:
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import pathlib, sys
    sys.path.insert(0, str(pathlib.Path.cwd().parent))

from jacopy.algebra.derivation import Act
from jacopy.brackets.base import BracketApply
from jacopy.brackets.schouten import sn as default_sn
from jacopy.calculus.hamiltonian_vf import hamiltonian_vf
from jacopy.calculus.poisson_axioms import (
    HamiltonianCyclicSnFormulaDefinition,
    PoissonAsHamiltonianDefinition,
)
from jacopy.core.expr import Neg, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.library.poisson import PoissonBracket
from jacopy.proof.expansion import ExpansionEngine

## 1. Kurulum — bivector + generic functions

Poisson bracket'ini soyut bir bivektör `π` üzerinde inşa ediyoruz. Üç
generic fonksiyon `f, g, h` SN-shifted dereceyle (–1, fonksiyonların 2-vektör
generator'ünün indirgenmiş gradasyonundaki kaymış derecesi) kayıt ediliyor:
bu, `graded_jacobi_obstruction`'ın Koszul işaret hesaplarını hatasız
yapabilmesi için gerekli.

In [2]:
reg = PropertyRegistry()

pi = Symbol("π");  reg.declare(pi, Graded(degree=1))
f = Symbol("f");   reg.declare(f,  Graded(degree=-1))
g = Symbol("g");   reg.declare(g,  Graded(degree=-1))
h = Symbol("h");   reg.declare(h,  Graded(degree=-1))

P = PoissonBracket.from_bivector(pi)
P_derived = P.derived  # the underlying DerivedBracket(sn, π)
P_derived

DerivedBracket('{·,·}_π')

## 2. LHS — fonksiyon-düzeyinde cyclic Poisson Jacobi

`graded_jacobi_obstruction` üç cyclic permütasyonun Koszul-işaretli
toplamını üretir. Fonksiyonların kaymış SN-derecesi `–1` olduğu için
parite çarpımları `1` çıkar ve her cyclic terim `Neg`-sarmalı olarak
gelir:

In [3]:
lhs = P_derived.graded_jacobi_obstruction(f, g, h, registry=reg)
print(lhs)

((-{·,·}_π(f, {·,·}_π(g, h))) + (-{·,·}_π(g, {·,·}_π(h, f))) + (-{·,·}_π(h, {·,·}_π(f, g))))


Üç çocuk, üç negatif `BracketApply` — cyclic permütasyonun her bir
yörüngesi yine cyclic döngünün bir kopyası:

In [4]:
for i, c in enumerate(lhs.children):
    print(f"[{i}] {c}")

[0] (-{·,·}_π(f, {·,·}_π(g, h)))
[1] (-{·,·}_π(g, {·,·}_π(h, f)))
[2] (-{·,·}_π(h, {·,·}_π(f, g)))


## 3. Aksiyomlar — Faz 13.E iki kuralı

**Aksiyom 2g-1 (PoissonAsHamiltonianDefinition).** Bizim `P_derived`
örneğimize sabitlenmiş: yalnızca `BracketApply(P_derived, ·, ·)` düğümlerini
yakalar ve `Act(X_f, g)` olarak yeniden yazar. Diğer derived bracket'leri
karıştırmaz.

**Aksiyom 2g-2 (HamiltonianCyclicSnFormulaDefinition).** 13.D'nin fonksiyon-
düzeyindeki kardeşi: cyclic `Act(X_a, Act(X_b, c))` üçlüsü → `BracketApply(sn,
π, π)`. Üç terimin polaritesi tutarlı olmalı (üçü de pozitif veya üçü de
`Neg`-sarmalı); RHS'deki SN handle aynı işareti taşır.

In [5]:
axiom_2g_1 = PoissonAsHamiltonianDefinition(P_derived)
axiom_2g_2 = HamiltonianCyclicSnFormulaDefinition(pi)

print("axiom 2g-1 :", axiom_2g_1.name)
print("axiom 2g-2 :", axiom_2g_2.name)

axiom 2g-1 : {f,g}_π = X_f(g) [{·,·}_π]
axiom 2g-2 : [π,π]_SN function-level formula


## 4. Engine — iki aksiyomla full kapanış

Engine `expand` çağrısı bottom-up olarak yürür: önce iç
`BracketApply(P_derived, g, h)` 2g-1 ile `Act(X_g, h)` olur, sonra dış
`BracketApply(P_derived, f, Act(X_g, h))` 2g-1 ile `Act(X_f, Act(X_g, h))`
olur. Üç cyclic terim aynı şekle indikten sonra 2g-2 cyclic üçlüyü tanır
ve `Neg(BracketApply(sn, π, π))` ile değiştirir.

In [6]:
engine = ExpansionEngine([axiom_2g_1, axiom_2g_2])
result, steps = engine.expand(lhs)
result

(-[·,·]_SN(π, π))

Kapanış doğrulaması — sonuç tam olarak `Neg(BracketApply([·,·]_SN, π,
π))`:

In [7]:
expected = Neg(BracketApply(default_sn, pi, pi))
print("result    :", result)
print("expected  :", expected)
print("match     :", result == expected)

result    : (-[·,·]_SN(π, π))
expected  : (-[·,·]_SN(π, π))
match     : True


İşaret yorumu: cyclic Jacobi LHS Koszul-işaretli olduğu için sonuç
`Neg(...)` taşıyor. Mutlak değer olarak engelin handle'ı evrensel
`BracketApply(sn, π, π)`; aynı düğüm form-düzeyinde 2f-deep notebook'unun
13.D aksiyomundan da çıkıyor — fonksiyon ve form yolları aynı obstruction
nesnesinde buluşuyor.

## 5. Adım listesi — proof transcript

Her adım hangi aksiyomun nereye uygulandığını gösteriyor. 2g-1 üç kez
(üç cyclic terimde, her birinde inner + outer için iki kez) ateşler;
2g-2 son adımda cyclic üçlüyü tanır:

In [8]:
for i, step in enumerate(steps):
    print(f"[{i:02d}] {step.rule}")

[00] {f,g}_π = X_f(g) [{·,·}_π]
[01] {f,g}_π = X_f(g) [{·,·}_π]
[02] {f,g}_π = X_f(g) [{·,·}_π]
[03] {f,g}_π = X_f(g) [{·,·}_π]
[04] {f,g}_π = X_f(g) [{·,·}_π]
[05] {f,g}_π = X_f(g) [{·,·}_π]
[06] [π,π]_SN function-level formula


## 6. 2f-deep ile karşılaştırma — neden fonksiyon tarafı temiz kapanır

Form tarafı (2f-deep) `KoszulBracket.expand`'ın her bir 1-formu Lie
derivative + d-of-pairing parçalarına ayırmasıyla başlar; engine `L_X`'in
opaklığını kırmak için ek bookkeeping kuralları (R-linearity-in-vector-field,
[L,d]=0 cascade'i, d²=0) ister. 2f-deep notebook'unun "kalan iş"
bölümü tam da bu rezidüyü betimliyor.

Fonksiyon tarafı bu açılımı atlar: Poisson bracket'i doğrudan Hamiltonian
operatöre çöker (2g-1), ardından cyclic Hamiltonian üçlüsü 2g-2'nin
örüntüsüne aynen oturur. Her iki yol da aynı `[π, π]_SN` handle'ında
buluşur — ama 2g-deep, sadece *iki* aksiyom (artı engine'in standart
bottom-up yürüyüşü) ile kapanır.

## 7. Bağımsız aksiyom davranışı

İki aksiyomun da kendi başlarına nasıl davrandığını izole etmek için
küçük örnekler:

In [9]:
# Axiom 2g-1 — single bracket collapse
sample = BracketApply(P_derived, f, g)
print("in :", sample)
print("out:", axiom_2g_1.rewrite(sample))

in : {·,·}_π(f, g)
out: X_f(g)


In [10]:
# Axiom 2g-2 — saf pozitif cyclic üçlü (residue olmadan)
def _ham_term(a, b, c):
    Xa = hamiltonian_vf(a, bivector=pi)
    Xb = hamiltonian_vf(b, bivector=pi)
    return Act(Xa, Act(Xb, c))

triple = Sum(
    _ham_term(f, g, h),
    _ham_term(g, h, f),
    _ham_term(h, f, g),
)
print("in :", triple)
print("out:", axiom_2g_2.rewrite(triple))

in : (X_f(X_g(h)) + X_g(X_h(f)) + X_h(X_f(g)))
out: [·,·]_SN(π, π)


## 8. Sonuç

Faz 13 fonksiyon-tarafı kapanışı tamamlandı:

* **2g-1** Poisson bracket'ini Hamiltonian eyleme indirger.
* **2g-2** cyclic Hamiltonian üçlüsünü evrensel SN handle'ına indirger.

İki aksiyom, hiçbir hazır teorem cite etmeden, fonksiyon-düzeyinde
`Σ_cyc {f, {g, h}_π}_π = ½⟨[π,π]_SN, df∧dg∧dh⟩` özdeşliğinin engine-derived
ispatını üretir. Form ve fonksiyon yolları artık aynı `BracketApply(sn,
π, π)` düğümünde buluşuyor — Faz 13'ün hedeflediği iki yönlü senkronizasyon
sağlanmış oldu.